# Application tests for the lab computer

Run the setup cell first, then one numbered cell at a time. The setup cell chooses the image size, source and destination. Each test saves a JSON report in `CONFIG.output_dir`. `EXECUTED` means the code finished; review the pass flags and pending measurements in the report.

Use the project's `.venv` as the notebook kernel. The notebook is in `application tests`; its setup cell searches upward for the project root. If notebook support is unavailable, run one section from PowerShell in the project root, for example `./.venv/Scripts/python.exe scripts/lab_computer_acceptance.py --section 3 --source mock`.

Start with `CONFIG.source = "mock"`. For physical camera tests, close CamWare and our camera GUI, check the ROI/exposure/shutter settings, then set `CONFIG.source = "physical"`. Camera tests close their handle after each section. The 455 fps setting is a manufacturer reference for 2560 × 128 Global Shutter, not a measured rate on this PC. Do not use Run All: section 5 plans about 109 GB of raw data with the current ROI. Section 8 is optional and deferred.


In [4]:
from pathlib import Path
import sys

# Start the notebook in the repository root or scripts/ directory.
start = Path.cwd()
ROOT = next((p for p in (start, *start.parents) if (p / "src/hhg_control").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the LabSoftware - Codex checkout")
sys.path.insert(0, str(ROOT))
from scripts import lab_computer_acceptance as lab

CONFIG = lab.Settings()
# Edit these before running the remaining cells:
CONFIG.source = "mock"  # Set to "physical" on the main PC after reviewing settings.
CONFIG.output_dir = ROOT / "data/test_data/lab_acceptance"  # e.g. Path("D:/CameraTests")
CONFIG.expected_serial = None  # Enter the serial from the camera label before physical connection.
CONFIG.usb_port_note = "FILL IN: USB controller, port and cable"
CONFIG.roi = (0, 1016, 2560, 1144)
CONFIG.exposure_s = 0.001
CONFIG.readout = lab.ReadoutMode.GLOBAL_SHUTTER
CONFIG.target_fps = 455.0  # Storage load/reference; replace with measured rate.
CONFIG.camware_fps = None  # Measured CamWare memory-recording rate at matched settings.
CONFIG.sustained_seconds = 300.0
CONFIG.max_section_raw_gb = 120.0
CONFIG.trigger_bench_ready = False  # Section 8 remains deferred.
print(CONFIG)


Settings(source='mock', output_dir=WindowsPath('c:/Users/omerp/Documents/לימודים/תואר שני/Research/Software work/LabSoftware - Codex/data/test_data/lab_acceptance'), expected_serial=None, usb_port_note='FILL IN: USB controller, port and cable', roi=(0, 1016, 2560, 1144), exposure_s=0.001, readout=<ReadoutMode.GLOBAL_SHUTTER: 2>, target_fps=455.0, camware_fps=None, speed_fraction=0.9, storage_margin=1.3, batch_size=4, short_frames=(20, 1000), speed_frames=10000, repeats=3, sustained_seconds=300.0, max_section_raw_gb=120.0, run_offline_tests=True, data_files=(), trigger_bench_ready=False, trigger_frames=100, observed_trigger_pulses=None)


## 1. Computer and storage readiness

This runs the offline software tests, then feeds synthetic frames through the production HDF5 writer on your chosen drive. It measures storage capacity and a paced replay, including whether saved frame markers stay in order. It does not connect to the camera or test USB transfer. The full software test result is saved in a log; a GUI layout failure is recorded while the independent storage measurements still run.

Set `CONFIG.output_dir` to the actual acquisition drive. Watch **Task Manager → Performance → Disk** during a longer write; short results can benefit from caches. At 2560 × 128 and the 455 fps reference, raw data would arrive at **298.19 MB/s**. The proposed 30% writer margin is **387.65 MB/s**. These are targets to compare with measurements, not proof that the physical camera reaches 455 fps. The paced replay must retain every ID and image marker; review its maximum lateness.


In [ ]:
REPORT_1 = lab.execute(1, CONFIG)


## 2. Camera identity and settings

This connects twice and reads back the camera model, hardware serial, ROI, exposure, shutter and trigger mode. It applies the settings in `CONFIG`, so close CamWare and the camera GUI first. Changing shutter mode may reboot the camera.

**Global Shutter is mode value 2.** `CONFIG.expected_serial` means the individual camera's serial number, not the shutter mode. Enter the serial from the camera label before any physical connection. Both connections should report the same identity and settings. Separately confirm in the GUI that the controls remain usable after a mode change and reconnect.


In [ ]:
REPORT_2 = lab.execute(2, CONFIG)


## 3. Short saved acquisitions

This saves 20 frames, then 1,000 frames through the same HDF5 writer used by the application. Check each result for exact requested and written counts, `uint16` image size, consecutive camera frame numbers, and a complete `.h5` file. A `.h5.partial` file marks an interrupted step. Look at the first, middle and last images to confirm the intended scene, orientation and ROI coverage before longer runs.


In [ ]:
REPORT_3 = lab.execute(3, CONFIG)


## 4. Camera download and saving speed

**CamWare first-time procedure** (use the same PC, USB port and camera as our application):

1. Close our GUI and any notebook camera connection. Open **pco.camware** and select the physical camera in **Camera Overview**.
2. In **Camera Properties**, choose **Global Shutter**, **Auto Sequence** trigger, **1 ms exposure**, and a centered **2560 × 128 ROI**. If the ROI window asks for one-based inclusive bounds, use Left 1, Right 2560, Top 1017, Bottom 1144. Confirm the displayed image size is 2560 × 128. Do not add an FPS limit or delay that differs from our settings.
3. In **Acquisition → Memory Allocation Dialog**, start with 1,000 images (about 0.66 GB raw). Try 10,000 (about 6.55 GB raw) only if CamWare allows it with available RAM.
4. Choose **Record Sequence**, not Live Preview. Note the actual number of frames and time from starting until recording finishes; calculate `frames / seconds`. Repeat three times. Prefer the longer run for manual timing. Playback speed and a configured FPS value are not measured capture rates.
5. Enter the matched **memory-recording** rate in `CONFIG.camware_fps`, close CamWare, then run this notebook cell. The notebook reads camera frames without saving, then measures the complete saved scan, each repeated three times. You can time the same scan through our GUI separately.

The proposed check compares our **capture-only** rate with CamWare's memory-recording rate (target: at least 90%). The saved rate is reported separately because HDF5 writing and closing take time. Missing frames or recorder errors fail a run regardless of average speed. A slow capture-only rate points toward camera/USB/SDK reading; a fast capture-only rate with slow saving points toward the writer or drive. The [Excelitas CamWare manual](https://filehub.excelitas.com/api/public/dl/r6yTVmC5/PCO_MA_PCOCAMWARE.pdf) describes Camera Properties, Record Sequence and the Memory Allocation Dialog.


In [ ]:
REPORT_4 = lab.execute(4, CONFIG)


## 5. Repeated and sustained reliability

This saves three large steps and one long continuous recording, checking every completed file for counts and gaps. Watch Task Manager's memory, CPU and disk activity during the run and note whether performance or memory use worsens after warm-up. A frame count is calculated from `CONFIG.target_fps × CONFIG.sustained_seconds`; the observed acquisition span in the report tells you how long it actually lasted.

At 2560 × 128, the default section plans about **109 GB raw data** and requires about **164 GB free** under its space preflight. Reduce `CONFIG.sustained_seconds` for a preliminary trial if needed. Compare sustained rate with section 4; a drop greater than 10% deserves investigation. Every step must have exact counts, consecutive frame IDs, and no FIFO or writer-queue errors. Scan setpoints in the current application label metadata; they do not move a physical stage.


In [ ]:
REPORT_5 = lab.execute(5, CONFIG)


## 6. Failure recovery and GUI use

The automatic portion always uses a **mock**, even if `CONFIG.source` says `physical`. It simulates an abort and a disk writer error. Both should leave visibly incomplete `.h5.partial` files, while a following recording succeeds without restarting Python.

The remaining checks are manual: cycle Live → Stop → Scan → Stop → Live; try Stop during a short physical scan; verify a useful error for an unavailable destination; reconnect the camera only while recording is stopped; and check that the GUI never allows two owners of the camera. No failed step should appear as a completed `.h5` file. Record how long Stop and normal controls respond. Changing shutter mode has a separate expected reboot delay.


In [ ]:
REPORT_6 = lab.execute(6, CONFIG)


## 7. Verify a saved scan and its optional copy in Python

Run this cell after section 3. By default, it checks the **latest short scan**: the file is complete, image and metadata counts agree, frame IDs have no gaps, every raw `uint16` pixel can be read, and the ROI and stored settings agree. It reports first/middle/last frame statistics in raw ADU and computes SHA-256 fingerprints. It does not connect to the camera or run MATLAB.

To check an archived copy, edit the code cell immediately below to set `CONFIG.data_files` to `(Path("D:/.../original.h5"), Path("E:/.../copy.h5"))`. Run the cell once; it compares complete file bytes, pixels, metadata and settings, and fails if they differ. A pair must be an original and its actual copy. Leave `CONFIG.data_files = ()` for automatic selection. These checks validate data storage and copying; judging dark noise, saturation and physical signal quality needs separate calibration and scene review.


In [ ]:
# Leave () to verify the latest short scan from section 3.
# To compare an original and an archive copy, uncomment and edit this line:
# CONFIG.data_files = (Path("D:/CameraTests/original.h5"), Path("E:/Archive/copy.h5"))
REPORT_7 = lab.execute(7, CONFIG)


## 8. External trigger accounting — deferred

Leave `CONFIG.trigger_bench_ready = False` for now. The cell records **DEFERRED** and opens no camera. Later, this test requires a verified pulse source and an independent counter or oscilloscope. Camera frame IDs alone cannot reveal laser pulses the camera never accepted.


In [ ]:
REPORT_8 = lab.execute(8, CONFIG)
